# Financial Time Series Data Exploration with TTSTL

This notebook demonstrates data exploration and feature engineering for financial time series data using the TTSTL (Two Time Scale Transfer Learning) package utilities.

## Objectives
1. Load and explore limit order book (LOB) data
2. Preprocess data using TTSTL utilities
3. Visualize time series patterns and features
4. Prepare data for TTSTL modeling

In [ ]:
# Import Required Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Add src to path for local development
sys.path.append(str(Path().resolve().parent / "src"))

print("Libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")
print(f"Python version: {sys.version}")

In [ ]:
# Import TTSTL utilities
try:
    from ttstl.utils import (
        preprocess_mbo_data_cnn_lstm,
        download_from_bucket,
        import_to_pandas
    )
    from ttstl.backbone import CNNLSTMDataset
    print("✅ TTSTL package imported successfully!")
except ImportError as e:
    print(f"❌ Error importing TTSTL: {e}")
    print("Please install the package first: pip install -e .")
    
# Import additional analysis libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

## 1. Data Generation and Loading

Since we may not have access to real limit order book data, let's generate synthetic financial time series data that mimics the structure of real market microstructure data.

In [ ]:
# Generate synthetic limit order book data
def generate_synthetic_lob_data(n_samples=10000, start_price=100.0):
    """
    Generate synthetic limit order book data for exploration
    """
    np.random.seed(42)
    
    # Time series
    timestamps = pd.date_range('2024-01-01', periods=n_samples, freq='100ms')
    
    # Price evolution (random walk with drift)
    price_changes = np.random.normal(0, 0.01, n_samples)
    prices = start_price + np.cumsum(price_changes)
    
    # Order flow features (synthetic)
    data = {
        'timestamp': timestamps,
        'micro': prices,  # Mid price
        'bid_price_1': prices - np.random.exponential(0.01, n_samples),
        'ask_price_1': prices + np.random.exponential(0.01, n_samples),
        'bid_volume_1': np.random.exponential(100, n_samples),
        'ask_volume_1': np.random.exponential(100, n_samples),
        'trade_volume': np.random.exponential(50, n_samples),
        'order_flow_imbalance': np.random.normal(0, 10, n_samples),
        'volatility_proxy': np.abs(price_changes) * 100,
        'spread': np.random.exponential(0.005, n_samples),
    }
    
    # Add some order flow features
    for level in range(2, 6):  # Levels 2-5 of the order book
        data[f'bid_price_{level}'] = data['bid_price_1'] - (level-1) * np.random.exponential(0.002, n_samples)
        data[f'ask_price_{level}'] = data['ask_price_1'] + (level-1) * np.random.exponential(0.002, n_samples)
        data[f'bid_volume_{level}'] = np.random.exponential(80, n_samples)
        data[f'ask_volume_{level}'] = np.random.exponential(80, n_samples)
    
    df = pd.DataFrame(data)
    return df

# Generate data
print("Generating synthetic limit order book data...")
lob_data = generate_synthetic_lob_data(n_samples=50000)
print(f"Generated {len(lob_data)} observations")
print(f"Time range: {lob_data['timestamp'].min()} to {lob_data['timestamp'].max()}")
print(f"Data shape: {lob_data.shape}")

# Display first few rows
lob_data.head()

## 2. Basic Data Exploration

Let's examine the basic characteristics of our synthetic limit order book data.

In [ ]:
# Basic data information
print("Dataset Information:")
print("=" * 50)
print(f"Shape: {lob_data.shape}")
print(f"Memory usage: {lob_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Data types:\n{lob_data.dtypes.value_counts()}")

print("\nColumn names:")
print(list(lob_data.columns))

print("\nMissing values:")
missing_values = lob_data.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.any() else "No missing values")

# Basic statistics
print("\nDescriptive Statistics:")
print("=" * 50)
numeric_cols = lob_data.select_dtypes(include=[np.number]).columns
lob_data[numeric_cols].describe()

In [ ]:
# Plot price evolution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price evolution
axes[0, 0].plot(lob_data['timestamp'][:5000], lob_data['micro'][:5000], alpha=0.8)
axes[0, 0].set_title('Mid Price Evolution (First 5000 points)')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Price')
axes[0, 0].grid(True, alpha=0.3)

# Price returns
returns = lob_data['micro'].pct_change().dropna()
axes[0, 1].hist(returns, bins=50, alpha=0.7, density=True)
axes[0, 1].set_title('Price Returns Distribution')
axes[0, 1].set_xlabel('Returns')
axes[0, 1].set_ylabel('Density')
axes[0, 1].grid(True, alpha=0.3)

# Spread over time
axes[1, 0].plot(lob_data['timestamp'][:5000], lob_data['spread'][:5000], alpha=0.6)
axes[1, 0].set_title('Bid-Ask Spread (First 5000 points)')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Spread')
axes[1, 0].grid(True, alpha=0.3)

# Volume analysis
axes[1, 1].scatter(lob_data['bid_volume_1'][:1000], lob_data['ask_volume_1'][:1000], 
                   alpha=0.5, s=10)
axes[1, 1].set_title('Bid vs Ask Volume (First 1000 points)')
axes[1, 1].set_xlabel('Bid Volume')
axes[1, 1].set_ylabel('Ask Volume')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print some key statistics
print("Key Statistics:")
print(f"Price volatility (std): {returns.std():.6f}")
print(f"Average spread: {lob_data['spread'].mean():.6f}")
print(f"Max price: {lob_data['micro'].max():.2f}")
print(f"Min price: {lob_data['micro'].min():.2f}")

## 3. Data Preprocessing with TTSTL Utilities

Now let's use the TTSTL preprocessing utilities to prepare the data for machine learning. We'll save our synthetic data first and then process it using the package utilities.

In [ ]:
# Save synthetic data to parquet format for preprocessing
data_path = "../data/synthetic_lob_data.parquet"
os.makedirs("../data", exist_ok=True)

# Convert timestamp to numeric for parquet compatibility
lob_data_save = lob_data.copy()
lob_data_save['timestamp'] = lob_data_save['timestamp'].astype('int64') // 10**9  # Convert to seconds

lob_data_save.to_parquet(data_path)
print(f"Data saved to {data_path}")

# Now use TTSTL preprocessing utilities
try:
    # Preprocess the data using TTSTL utilities
    delta = 0.1  # 100ms sampling (0.1 seconds)
    quantile = 0.005  # Winsorization quantile
    
    print(f"Preprocessing data with delta={delta}, quantile={quantile}")
    features, targets = preprocess_mbo_data_cnn_lstm(
        path=data_path,
        delta=delta,
        quantile=quantile
    )
    
    print(f"✅ Preprocessing successful!")
    print(f"Features shape: {features.shape}")
    print(f"Targets shape: {targets.shape}")
    
    # Display processed features
    print("\nProcessed Features (first 5 rows):")
    print(features.head())
    
    print("\nProcessed Targets (first 5 rows):")
    print(targets.head())
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    print("This might be due to missing implementation in the preprocessing function.")
    print("Let's create simplified features manually for now.")

In [ ]:
# Manual feature engineering for exploration
def create_features_manually(df, horizons=[1, 5, 10]):
    """
    Create features manually for exploration purposes
    """
    features_df = df.copy()
    
    # Select order flow features (columns containing bid/ask info)
    of_columns = [col for col in df.columns if any(x in col.lower() for x in ['bid', 'ask', 'volume', 'spread', 'flow'])]
    
    # Standardize the order flow features
    scaler = StandardScaler()
    features_df[of_columns] = scaler.fit_transform(features_df[of_columns])
    
    # Create target variables (price changes over different horizons)
    targets_dict = {}
    for h in horizons:
        # Calculate percentage change over horizon h
        future_price = df['micro'].shift(-h)
        current_price = df['micro']
        pct_change = ((future_price - current_price) / current_price) * 1000  # multiply by 1000 for better scale
        targets_dict[f'target_h{h}'] = pct_change
    
    targets_df = pd.DataFrame(targets_dict)
    
    # Remove timestamp for feature matrix
    feature_columns = [col for col in features_df.columns if col != 'timestamp']
    features_final = features_df[feature_columns]
    
    return features_final, targets_df

# Create features manually
print("Creating features manually...")
features_manual, targets_manual = create_features_manually(lob_data)

print(f"Manual features shape: {features_manual.shape}")
print(f"Manual targets shape: {targets_manual.shape}")

print("\nFeature columns:")
print(list(features_manual.columns))

print("\nTarget columns:")
print(list(targets_manual.columns))

# Display statistics
print("\nFeature statistics:")
print(features_manual.describe())

print("\nTarget statistics:")
print(targets_manual.describe())

## 4. Feature Analysis and Visualization

Let's analyze the relationships between features and examine the target variable distributions.

In [ ]:
# Feature correlation analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Feature correlation heatmap
correlation_matrix = features_manual.corr()
mask = np.triu(np.ones_like(correlation_matrix))

im = axes[0, 0].imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0, 0].set_title('Feature Correlation Matrix')
axes[0, 0].set_xticks(range(len(correlation_matrix.columns)))
axes[0, 0].set_yticks(range(len(correlation_matrix.columns)))
axes[0, 0].set_xticklabels(correlation_matrix.columns, rotation=45, ha='right')
axes[0, 0].set_yticklabels(correlation_matrix.columns)
plt.colorbar(im, ax=axes[0, 0])

# 2. Target distribution for different horizons
targets_clean = targets_manual.dropna()
for i, col in enumerate(targets_clean.columns):
    axes[0, 1].hist(targets_clean[col], bins=50, alpha=0.6, label=col, density=True)
axes[0, 1].set_title('Target Distributions by Horizon')
axes[0, 1].set_xlabel('Price Change (‰)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Feature importance via correlation with targets
feature_target_corr = []
for feature in features_manual.columns:
    corr_values = []
    for target in targets_manual.columns:
        # Calculate correlation, handling NaN values
        combined_data = pd.concat([features_manual[feature], targets_manual[target]], axis=1).dropna()
        if len(combined_data) > 0:
            corr = combined_data.corr().iloc[0, 1]
            corr_values.append(abs(corr))
        else:
            corr_values.append(0)
    feature_target_corr.append(np.mean(corr_values))

feature_importance_df = pd.DataFrame({
    'feature': features_manual.columns,
    'importance': feature_target_corr
}).sort_values('importance', ascending=True)

axes[1, 0].barh(range(len(feature_importance_df)), feature_importance_df['importance'])
axes[1, 0].set_yticks(range(len(feature_importance_df)))
axes[1, 0].set_yticklabels(feature_importance_df['feature'])
axes[1, 0].set_title('Feature-Target Correlation (Average)')
axes[1, 0].set_xlabel('Average Absolute Correlation')

# 4. PCA analysis
pca = PCA()
features_clean = features_manual.dropna()
if len(features_clean) > 0:
    pca_result = pca.fit_transform(features_clean)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    
    axes[1, 1].plot(range(1, len(cumvar) + 1), cumvar, 'bo-')
    axes[1, 1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
    axes[1, 1].set_title('PCA Cumulative Explained Variance')
    axes[1, 1].set_xlabel('Number of Components')
    axes[1, 1].set_ylabel('Cumulative Explained Variance')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Feature Analysis Summary:")
print("=" * 50)
print(f"Number of features: {len(features_manual.columns)}")
print(f"Top 5 most important features:")
for i, row in feature_importance_df.tail().iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

if len(features_clean) > 0:
    print(f"\nPCA Analysis:")
    print(f"  Components for 95% variance: {np.argmax(cumvar >= 0.95) + 1}")
    print(f"  Components for 90% variance: {np.argmax(cumvar >= 0.90) + 1}")

## 5. Data Preparation for TTSTL Modeling

Now let's prepare the data for use with TTSTL models, including sequence creation for the CNN-LSTM backbone.

In [ ]:
# Prepare sequences for CNN-LSTM training
def create_sequences(features, targets, sequence_length=50):
    """
    Create sequences for time series modeling
    """
    # Remove rows with NaN values
    combined_data = pd.concat([features, targets], axis=1).dropna()
    features_clean = combined_data[features.columns]
    targets_clean = combined_data[targets.columns]
    
    sequences = []
    sequence_targets = []
    
    for i in range(len(features_clean) - sequence_length):
        seq = features_clean.iloc[i:i+sequence_length].values
        target = targets_clean.iloc[i+sequence_length].values
        sequences.append(seq)
        sequence_targets.append(target)
    
    return np.array(sequences), np.array(sequence_targets)

# Create sequences
sequence_length = 50
print(f"Creating sequences with length {sequence_length}...")

X_sequences, y_sequences = create_sequences(features_manual, targets_manual, sequence_length)

print(f"✅ Sequences created successfully!")
print(f"X_sequences shape: {X_sequences.shape}")
print(f"y_sequences shape: {y_sequences.shape}")

# Split into train/validation/test
train_size = int(0.7 * len(X_sequences))
val_size = int(0.15 * len(X_sequences))

X_train = X_sequences[:train_size]
y_train = y_sequences[:train_size]
X_val = X_sequences[train_size:train_size+val_size]
y_val = y_sequences[train_size:train_size+val_size]
X_test = X_sequences[train_size+val_size:]
y_test = y_sequences[train_size+val_size:]

print(f"\nData splits:")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

# Visualize a sample sequence
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot first few features of a sample sequence
sample_idx = 0
feature_subset = X_train[sample_idx, :, :5]  # First 5 features
for i in range(feature_subset.shape[1]):
    axes[0].plot(feature_subset[:, i], label=f'Feature {i+1}', alpha=0.7)
axes[0].set_title(f'Sample Sequence Features (Sample {sample_idx})')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Normalized Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot corresponding targets
target_names = list(targets_manual.columns)
for i, target_name in enumerate(target_names):
    axes[1].bar(i, y_train[sample_idx, i], label=target_name, alpha=0.7)
axes[1].set_title(f'Corresponding Targets (Sample {sample_idx})')
axes[1].set_xlabel('Target Horizon')
axes[1].set_ylabel('Price Change (‰)')
axes[1].set_xticks(range(len(target_names)))
axes[1].set_xticklabels(target_names)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Summary and Next Steps

### Data Exploration Summary

We have successfully explored financial time series data and prepared it for TTSTL modeling:

1. **Generated synthetic limit order book data** with realistic characteristics
2. **Explored basic statistics** and visualized key features
3. **Created engineered features** from order flow information
4. **Analyzed feature correlations** and target variable distributions
5. **Prepared sequential data** for CNN-LSTM backbone training

### Key Findings

- The synthetic data exhibits realistic financial time series characteristics
- Order flow features show varying correlations with price change targets
- Different prediction horizons show different target distributions
- The data is ready for two-scale modeling approach

### Next Steps

1. **Implement TTSTL Components**: 
   - Complete the backbone CNN-LSTM implementation
   - Develop the interface for change detection
   - Create the adapter for local adaptation

2. **Model Training**:
   - Train the backbone model on historical data
   - Implement the two-scale prediction mechanism
   - Test adaptation capabilities

3. **Evaluation**:
   - Compare TTSTL performance against baseline models
   - Analyze adaptation frequency and effectiveness
   - Validate on out-of-sample data

### Data Ready for TTSTL

The prepared data is now ready for use with the TTSTL framework:
- **Features**: Standardized order flow features
- **Targets**: Multi-horizon price change predictions  
- **Sequences**: Time series sequences for neural network training
- **Splits**: Train/validation/test splits for proper evaluation